# Task 2 — Document Comparison (Text vs. Image)
**NewRocket Case Study | Hemanth Chebiyam**

---

## 1. Overview

In this notebook, we compare the similarity between a text-based document and an image-based document (e.g., a documentation screenshot or infographic). Our pipeline uses **OCR (Optical Character Recognition)** to extract text from images, followed by **Lexical** and **Semantic** similarity analysis.

### Methodology
1. **OCR Extraction**: Using `pytesseract` to retrieve text from the image.
2. **Lexical Similarity**: TF-IDF Vectorization + Cosine Similarity (keyword-based).
3. **Semantic Similarity**: Sentence-Transformer embeddings + Cosine Similarity (meaning-based).

In [ ]:
import os
import pytesseract
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings('ignore')

# Note: Ensure Tesseract-OCR is installed and added to your PATH or set the executable path below:
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

## 2. Defining Similarity Functions

We define two functions to measure different aspects of document similarity.

In [ ]:
def get_lexical_similarity(text1, text2):
    """Keyword-based similarity using TF-IDF."""
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])
    return cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

def get_semantic_similarity(text1, text2, model):
    """Meaning-based similarity using Sentence-Transformer embeddings."""
    embeddings = model.encode([text1, text2])
    return cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

# Load semantic model (pre-trained BERT-based model)
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

## 3. Comparison Pipeline

We pull together the OCR and similarity metrics into a single comparison function.

In [ ]:
def compare_documents(text_doc_path, image_doc_path):
    # 1. Load Text Document Content
    with open(text_doc_path, 'r', encoding='utf-8') as f:
        text_content = f.read()
    
    # 2. Extract Text from Image using OCR
    image_content = pytesseract.image_to_string(Image.open(image_doc_path))
    
    # 3. Compute Similarity Scores
    lex_score = get_lexical_similarity(text_content, image_content)
    sem_score = get_semantic_similarity(text_content, image_content, semantic_model)
    
    print("--- Comparison Results ---")
    print(f"Lexical Similarity (TF-IDF): {lex_score:.4f}")
    print(f"Semantic Similarity (Transformers): {sem_score:.4f}")
    print("---------------------------")
    return lex_score, sem_score

## 4. Test Case Examples

### Case 1: ServiceNow Incident KB Documentation
* **Source**: ServiceNow Docs [Incident Management](https://docs.servicenow.com/bundle/vancouver-it-service-management/page/product/incident-management/concept/c_IncidentManagement.html)
* **Comparison**: Official text description vs. Screenshot of the documentation page.

In [ ]:
# Example Usage (Placeholders - ensure files exist in /task2_data/)
# compare_documents('task2_data/incident_kb.txt', 'task2_data/incident_screenshot.png')

### Case 2: Flow Designer Workflow
* **Source**: ServiceNow Community flow diagram samples.
* **Comparison**: A textual list of steps in a password-reset flow vs. A screenshot of the Flow Designer canvas showing the same flow.

In [ ]:
# compare_documents('task2_data/pwd_reset_flow.txt', 'task2_data/flow_designer_diag.png')

## 5. Notes on Limitations & Extensions

* **OCR Dependency**: This comparison relies solely on textual representation. If an image contains mostly visual symbols (like a logo or a generic drawing) without text, the similarities will be low.
* **Future Extension (Multimodal)**: For deeper analysis (e.g., matching a description of a 'black dog' to a photograph of a dog), one would need a multimodal embedding model like **OpenAI's CLIP**. CLIP embeds both text and imagery into the same vector space, allowing for direct comparison without intermediate OCR.